# Train Quantile Regression cho nhiều mức tin cậy

## Vì sao cần

Bộ model quantile ban đầu **chỉ có 3 phân vị**: `q05` · `q50` · `q95` — đủ cho mức tin cậy **90%**.

Muốn so sánh **3 phương pháp × 3 mức tin cậy** thì cần thêm:

| Mức tin cậy | Cần phân vị |
|---|---|
| 90% | `q05` – `q95` ✅ đã có |
| **80%** | `q10` – `q90` ⬅️ **thiếu** |
| **70%** | `q15` – `q85` ⬅️ **thiếu** |

## Notebook này làm gì

Train LightGBM `objective="quantile"` cho **7 phân vị** × 3 tháng = **21 model**, lưu lại và sinh
dự đoán trên cả `calibration` và `test`.

⏱️ **Thời gian chạy ~5 phút.**

## Kiểm chứng

Mục 3 so `q05`/`q50`/`q95` mới train với bộ của lần chạy trước. Đây là kiểm tra **ổn định giữa
các lần chạy** — LightGBM có tính ngẫu nhiên dù đã cố định `random_state`. Lệch vài phần trăm
điểm là bình thường; lệch lớn ⇒ feature contract hoặc tham số đã đổi, phải xem lại.

> **Về lỗ hổng tái lập:** bộ 3 phân vị `qr_pred_*.parquet` và `quantile_models.joblib` trước đây
> do một script ngoài repo sinh ra. Nay `train/07_sinh_du_lieu_UQ.ipynb` dựng lại chúng từ chính
> bộ 7 phân vị của notebook này (lấy tập con `q05`/`q50`/`q95`) — chạy **sau** notebook này.

In [1]:
import warnings, time, joblib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor

CAT = ["service_name", "pickup_location_name", "dropoff_location_name", "weather_main"]
D_NUM = ["quote_distance", "quote_duration", "gio_vn", "latest_observed_price",
         "history_60m_price_mean", "history_60m_price_std",
         "history_60m_price_slope_per_minute", "latest_observed_quote_distance",
         "latest_observed_quote_duration", "actual_observation_age_minutes"]
FEATS = CAT + D_NUM

# 7 phan vi: du cho ca 3 muc tin cay 70 / 80 / 90
ALPHAS = [0.05, 0.10, 0.15, 0.50, 0.85, 0.90, 0.95]
MUC_PV = {0.90: (0.05, 0.95), 0.80: (0.10, 0.90), 0.70: (0.15, 0.85)}

QDIR = Path("../QuantileLGBM"); QDIR.mkdir(exist_ok=True)
EVAL = Path("../evaluation")

COLS = list(dict.fromkeys(FEATS + ["target_shown_price", "evaluation_month",
                                   "split", "requested_lag_minutes"]))
df = pd.read_parquet("../../data/hcm_train_ready.parquet", columns=COLS)
for c in CAT:
    df[c] = df[c].astype("category")

print(f"Nap {len(df):,} dong | {len(FEATS)} feature ({len(CAT)} cat + {len(D_NUM)} num)")
print(f"Phan vi se train: {ALPHAS}")
print()
print(df.groupby("split", observed=True).size().to_string())

Nap 6,897,051 dong | 14 feature (4 cat + 10 num)
Phan vi se train: [0.05, 0.1, 0.15, 0.5, 0.85, 0.9, 0.95]



split
calibration     615908
test            864360
train          4641799
validation      774984


## 1. Train 7 phân vị × 3 tháng

In [2]:
def tao_model(alpha):
    return LGBMRegressor(objective="quantile", alpha=alpha,
                         n_estimators=400, learning_rate=0.05, num_leaves=63,
                         min_child_samples=40, subsample=0.9, subsample_freq=1,
                         colsample_bytree=0.9, reg_lambda=1.0,
                         random_state=42, n_jobs=-1, verbose=-1)


thangs = sorted(df.evaluation_month.unique())
models = {}
t_tong = time.time()

for th in thangs:
    tr = df[(df.evaluation_month == th) & (df.split == "train")]
    X = tr[FEATS]
    y = np.log(tr.target_shown_price.values)
    print(f"\n[{th}]  n_train = {len(tr):,}")
    for a in ALPHAS:
        t0 = time.time()
        m = tao_model(a).fit(X, y, categorical_feature=CAT)
        models[(th, a)] = m
        print(f"   alpha={a:<5.2f} {time.time()-t0:5.0f}s")

joblib.dump(models, QDIR / "quantile_models_da_muc.joblib")
print(f"\nTONG: {(time.time()-t_tong)/60:.1f} phut | {len(models)} model")
print(f"Da luu -> {QDIR / 'quantile_models_da_muc.joblib'}")


[2026-01]  n_train = 1,544,286


   alpha=0.05     11s


   alpha=0.10     12s


   alpha=0.15     12s


   alpha=0.50     13s


...

   alpha=0.50     13s


   alpha=0.85     12s


   alpha=0.90     12s


   alpha=0.95     15s



TONG: 4.3 phut | 21 model
Da luu -> ..\QuantileLGBM\quantile_models_da_muc.joblib


## 2. Sinh dự đoán trên calibration và test

⚠️ **Thứ tự hàng:** gộp theo **tháng** để khớp với `uq_pred_*.parquet` và `qr_pred_*.parquet`.

In [3]:
COT_KEM = ["evaluation_month", "split", "requested_lag_minutes", "weather_main",
           "quote_distance", "gio_vn"]

for split in ["calibration", "test"]:
    phan = []
    for th in thangs:
        sub = df[(df.evaluation_month == th) & (df.split == split)]
        if len(sub) == 0:
            continue
        out = pd.DataFrame({
            "evaluation_month": sub.evaluation_month.values,
            "split": split,
            "requested_lag_minutes": sub.requested_lag_minutes.values,
            "weather_main": sub.weather_main.astype(str).values,
            "quang_duong": sub.quote_distance.values,
            "gio_vn": sub.gio_vn.values,
            "gia_that": sub.target_shown_price.values.astype("float64"),
        })
        for a in ALPHAS:
            ten = f"q{int(round(a*100)):02d}"
            out[ten] = np.exp(models[(th, a)].predict(sub[FEATS]))
        phan.append(out)
    pred = pd.concat(phan, ignore_index=True)
    f = EVAL / f"qr_pred_da_muc_{split}.parquet"
    pred.to_parquet(f, index=False)
    print(f"{split:12s}: {len(pred):>9,} dong -> {f.name}")

pred_test = pd.read_parquet(EVAL / "qr_pred_da_muc_test.parquet")
display(pred_test.head(3))

calibration :   615,908 dong -> qr_pred_da_muc_calibration.parquet


test        :   864,360 dong -> qr_pred_da_muc_test.parquet


,evaluation_month,split,requested_lag_minutes,weather_main,quang_duong,gio_vn,gia_that,q05,q10,q15,q50,q85,q90,q95
0,2026-01,test,5,Rain,5.708,7,106000.0,75335.037019,79667.181152,83015.555266,101057.351412,121072.586391,128401.907999,136053.475286
1,2026-01,test,10,Rain,5.708,7,106000.0,73861.333337,77581.892160,80506.492498,100456.581718,121291.064536,126509.843449,137210.258998
2,2026-01,test,15,Rain,5.708,7,106000.0,74644.298060,79482.753251,82192.156939,101931.803803,123306.582617,129358.242524,139199.991395


## 3. Kiểm chứng — so với bộ của lần chạy trước

`qr_pred_test.parquet` là sản phẩm của `train/07`, dựng từ bộ 7 phân vị của **lần chạy trước**
notebook này. Nên phép so dưới đây đo **độ ổn định giữa hai lần train** — không phải so với một
nguồn độc lập.

Lệch vài phần trăm điểm là nhiễu ngẫu nhiên của LightGBM. Lệch lớn ⇒ feature contract hoặc tham
số đã đổi, phải xem lại trước khi dùng.

In [4]:
cu = pd.read_parquet(EVAL / "qr_pred_test.parquet")
moi = pd.read_parquet(EVAL / "qr_pred_da_muc_test.parquet")
assert len(cu) == len(moi), f"So dong lech: cu {len(cu):,} vs moi {len(moi):,}"
assert np.allclose(cu.gia_that.values, moi.gia_that.values), "THU TU HANG KHONG KHOP!"
print(f"Khop hang: {len(cu):,} dong\n")

rows = []
for c in ["q05", "q50", "q95"]:
    a, b = cu[c].values, moi[c].values
    rows.append({"Phân vị": c,
                 "TB cũ": round(a.mean()), "TB mới": round(b.mean()),
                 "Chênh TB": f"{b.mean()/a.mean()-1:+.2%}",
                 "Tương quan": round(np.corrcoef(a, b)[0, 1], 4),
                 "MAE lệch": round(np.abs(a-b).mean())})
display(pd.DataFrame(rows))

# Coverage cua khoang 90% — cu vs moi
cov_cu = ((cu.gia_that >= cu.q05) & (cu.gia_that <= cu.q95)).mean()
cov_moi = ((moi.gia_that >= moi.q05) & (moi.gia_that <= moi.q95)).mean()
print(f"\nCoverage khoang q05-q95 (QR tho, chua hieu chinh):")
print(f"  Bo cu : {cov_cu:.2%}")
print(f"  Bo moi: {cov_moi:.2%}")
print(f"  Chenh : {abs(cov_moi-cov_cu)*100:.2f} diem")
if abs(cov_moi - cov_cu) < 0.02:
    print("\n  => KHOP. Bo moi dung duoc thay bo cu.")
else:
    print("\n  ⚠️ LECH DANG KE — xem lai feature contract / tham so.")

Khop hang: 864,360 dong



,Phân vị,TB cũ,TB mới,Chênh TB,Tương quan,MAE lệch
0,q05,87697,87787,+0.10%,0.9985,1006
1,q50,119972,119975,+0.00%,0.9990,1186
2,q95,163674,163569,-0.06%,0.9987,1850



Coverage khoang q05-q95 (QR tho, chua hieu chinh):
  Bo cu : 89.18%
  Bo moi: 89.09%
  Chenh : 0.09 diem

  => KHOP. Bo moi dung duoc thay bo cu.


## 4. Coverage thô của từng mức (chưa hiệu chỉnh conformal)

Đây là **QR thô** ở 3 mức. Chưa có bảo đảm — con số này cho thấy QR tự nó lệch bao nhiêu so với
danh mục.

In [5]:
rows = []
for muc, (lo_a, hi_a) in sorted(MUC_PV.items()):
    lo = f"q{int(round(lo_a*100)):02d}"
    hi = f"q{int(round(hi_a*100)):02d}"
    trong = (moi.gia_that >= moi[lo]) & (moi.gia_that <= moi[hi])
    rong = (moi[hi] - moi[lo]).mean()
    rows.append({"Mức danh mục": f"{muc:.0%}",
                 "Phân vị": f"{lo} – {hi}",
                 "Coverage đạt": f"{trong.mean():.2%}",
                 "Lệch": f"{(trong.mean()-muc)*100:+.2f} điểm",
                 "Độ rộng TB (đ)": round(rong)})
BANG = pd.DataFrame(rows)
display(BANG)
print("=> QR tho bam kha sat danh muc, nhung do la MAY, khong phai BAO DAM.")
print("   Notebook uncertainty/06 se hieu chinh bang conformal (CQR) de co bao dam.")

,Mức danh mục,Phân vị,Coverage đạt,Lệch,Độ rộng TB (đ)
0,70%,q15 – q85,68.88%,-1.12 điểm,47406
1,80%,q10 – q90,78.89%,-1.11 điểm,58772
2,90%,q05 – q95,89.09%,-0.91 điểm,75783


=> QR tho bam kha sat danh muc, nhung do la MAY, khong phai BAO DAM.
   Notebook uncertainty/06 se hieu chinh bang conformal (CQR) de co bao dam.


## Kết luận

### Artifact sinh ra

| File | Nội dung |
|---|---|
| `QuantileLGBM/quantile_models_da_muc.joblib` | 21 model — `{(tháng, alpha): LGBMRegressor}` |
| `evaluation/qr_pred_da_muc_calibration.parquet` | 7 phân vị trên calibration |
| `evaluation/qr_pred_da_muc_test.parquet` | 7 phân vị trên test |

### Bước tiếp theo

`model/uncertainty/04_SO_SANH.ipynb` — so **3 phương pháp × 3 mức tin cậy**:

| | 70% | 80% | 90% |
|---|---|---|---|
| Conformal chuẩn hoá | | | |
| QR thô | | | |
| CQR | | | |

### Liên quan

| File | Vai trò |
|---|---|
| `../uncertainty/04_SO_SANH.ipynb` | So 3 phương pháp ở mức 90% |
| `../uncertainty/01_conformal_chuan_hoa.ipynb` | Chọn mức tin cậy (chỉ conformal) |